In [12]:
import pandas as pd
import sqlite3

df_path = "../data/raw/chess_games.csv"
db_path = "../data/processed/chess.db"

df = pd.read_csv(df_path)
conn = sqlite3.connect(db_path)

# Create the table from the CSV columns (rated, turns, winner, opening_code, ...)
df.to_sql("games", conn, if_exists="replace", index=False)
conn.commit()
conn.close()

print("Data loaded successfully!")
print("Columns:", list(df.columns))








Data loaded successfully!
Columns: ['game_id', 'rated', 'turns', 'victory_status', 'winner', 'time_increment', 'white_id', 'white_rating', 'black_id', 'black_rating', 'moves', 'opening_code', 'opening_moves', 'opening_fullname', 'opening_shortname', 'opening_response', 'opening_variation']


## Stage 1 — SELECT · Stage 2 — GROUP BY : Q1-Q6

In [ ]:
# 1.How many total games? How many are rated?
import sqlite3

conn = sqlite3.connect("../data/processed/chess.db")
cursor = conn.cursor()
cursor.execute("""
SELECT
    COUNT(*) AS total_games,
    SUM(CASE WHEN rated = 1 THEN 1 ELSE 0 END) AS rated_games
FROM games;
""")
print("1-",cursor.fetchall())
conn.commit()
conn.close()








In [17]:

import sqlite3
#2.List all distinct victory_status values and their counts.
conn = sqlite3.connect("../data/processed/chess.db")
cursor = conn.cursor()
cursor.execute("""
SELECT
    victory_status,
    COUNT(*) AS count
FROM games
GROUP BY victory_status
ORDER BY count DESC;
""")
print("2-",cursor.fetchall())
conn.commit()
conn.close()


2- [('Resign', 11147), ('Mate', 6325), ('Out of Time', 1680), ('Draw', 906)]


In [18]:
#3.The 10 games with the most turns. Show game_id, winner, turns.
conn = sqlite3.connect("../data/processed/chess.db")
cursor = conn.cursor()
cursor.execute("""
SELECT
    game_id,
    winner,
    turns
FROM games
ORDER BY turns DESC
LIMIT 10;
""")
print("3-",cursor.fetchall())
conn.commit()
conn.close()


3- [(11555, 'White', 349), (13860, 'White', 349), (16387, 'Draw', 259), (4237, 'Draw', 255), (16646, 'Draw', 226), (15479, 'Draw', 222), (16944, 'Black', 222), (6777, 'Draw', 221), (13231, 'Black', 218), (13556, 'Draw', 216)]


In [20]:
#4.What is the win rate (%) for White, Black, and Draw across all games?
import sqlite3
conn=sqlite3.connect("../data/processed/chess.db")
cursor=conn.cursor()
cursor.execute("""
SELECT 
    winner,
    COUNT(*) * 100.0 /(SELECT COUNT(*) FROM games) AS win_rate_percentage 
FROM games
GROUP BY winner;
""")
print("4-",cursor.fetchall())
conn.commit()
conn.close()


4- [('Black', 45.403330342008175), ('Draw', 4.736264831987237), ('White', 49.86040482600459)]


In [21]:
#5. For each victory_status, what is the average and max number of turns? Sort highest
# avg first.
import sqlite3
conn=sqlite3.connect("../data/processed/chess.db")
cursor=conn.cursor()
cursor.execute("""
SELECT
victory_status,
AVG(turns) AS avg_turns,
MAX(turns) AS max_turns
FROM games
GROUP BY victory_status
ORDER BY avg_turns DESC;
""")
print("5-",cursor.fetchall())
conn.commit()
conn.close()

5- [('Draw', 83.78145695364239, 259), ('Out of Time', 72.74285714285715, 349), ('Mate', 65.41501976284584, 222), ('Resign', 53.91253251996053, 218)]


In [22]:
# #6.Which 5 opening_codes appear most frequently? Use HAVING to show only those
# with more than 500 games.
import sqlite3
conn=sqlite3.connect("../data/processed/chess.db")
cursor=conn.cursor()
cursor.execute("""
SELECT
    opening_code,
    COUNT(*) AS count
FROM games
GROUP BY opening_code
HAVING count > 500
ORDER BY count DESC
LIMIT 5;
""")
print("6-",cursor.fetchall())
conn.commit()
conn.close()


6- [('A00', 1007), ('C00', 844), ('D00', 739), ('B01', 716), ('C41', 691)]


## Stage 3 — JOINs & CTEs · Stage 4 — Window Functions

In [ ]:
#7. JOIN games to openings — find the 5 most played openings with their full name
import sqlite3
import pandas as pd
#create openings table
df_path = "../data/raw/chess_games.csv"
df = pd.read_csv(df_path)
conn=sqlite3.connect("../data/processed/chess.db")
openings_df = df[['opening_shortname', 'opening_fullname','opening_code']].drop_duplicates()
openings_df.to_sql('openings', conn, if_exists='append', index=False)
cursor=conn.cursor()

cursor.execute("""
SELECT 
    o.opening_fullname, 
    COUNT(g.game_id) AS total_played
FROM games g
JOIN openings o ON g.opening_code = o.opening_code
GROUP BY o.opening_fullname
ORDER BY total_played DESC
LIMIT 5;
    
""")

print("7-",cursor.fetchall())
conn.commit()
conn.close()

7- [('St. George Defense', 1455), ("Queen's Pawn Game: Chigorin Variation", 1262), ('Borg Defense: Borg Gambit', 1229), ('Ware Opening', 1007), ("Van't Kruijs Opening", 1007)]


In [34]:
#Create Players Table
import pandas as pd
import sqlite3
df_path = "../data/raw/chess_games.csv"
df=pd.read_csv(df_path)
conn=sqlite3.connect("../data/processed/chess.db")
cursor=conn.cursor()
conn.execute('PRAGMA foreign_keys = ON')
players_df=df[['white_id','black_id']].drop_duplicates()
players_df.to_sql('players',conn,if_exists='append',index=False)


17826

In [ ]:
#8-Q8 LEFT JOIN players to games: find players who have never appeared as white_id. 
import sqlite3
import pandas as pd

conn = sqlite3.connect("../data/processed/chess.db")
cursor = conn.cursor()
query = """
SELECT DISTINCT p.black_id
FROM players p
LEFT JOIN games g ON p.black_id = g.white_id
WHERE g.white_id IS NULL
"""

cursor.execute(query)
never_white_players = cursor.fetchall()
print("8 -", len(never_white_players))

conn.close()









8 - Players never played as White: 6197


In [ ]:
#9-Using a CTE: compute total wins per player (as white). Return top 5.
import sqlite3
import pandas as pd

conn = sqlite3.connect("../data/processed/chess.db")
cursor = conn.cursor()

query = """
WITH game_counts AS (
    
    SELECT white_id, COUNT(*) AS n 
    FROM games 
    WHERE winner = 'White'
    GROUP BY white_id
)

SELECT white_id, n 
FROM game_counts 
ORDER BY n DESC 
LIMIT 5;
"""

result = cursor.execute(query).fetchall()
print("9 -",result)

conn.close()

9 - [('taranga', 34), ('ssf7', 29), ('hassan1365416', 28), ('a_p_t_e_m_u_u', 25), ('1240100948', 22)]


In [ ]:
# 10- UNION CTE: combine white wins and black wins into one 'player_wins' table. Who has the
# most total wins?
import sqlite3
import pandas as pd
conn=sqlite3.connect("../data/processed/chess.db")
cursor=conn.cursor()
cursor.execute("""
WITH white_wins AS (
    SELECT white_id AS username, COUNT(*) AS wins 
    FROM games 
    WHERE winner = 'White' 
    GROUP BY white_id
),
black_wins AS (
    SELECT black_id AS username, COUNT(*) AS wins 
    FROM games 
    WHERE winner = 'Black' 
    GROUP BY black_id
),
player_wins AS (
    SELECT * FROM white_wins
    UNION ALL
    SELECT * FROM black_wins
)
SELECT username, SUM(wins) AS total_wins
FROM player_wins
GROUP BY username
ORDER BY total_wins DESC;
 
    """)

cursor.execute(query)
result=cursor.fetchall()
print("10 -",result)
conn.close()

10 - [('taranga', 34), ('ssf7', 29), ('hassan1365416', 28), ('a_p_t_e_m_u_u', 25), ('1240100948', 22)]


In [ ]:
#11 Window function: for each game, add a column showing what RANK each game holds for
# that white player by white_rating (highest rating = rank 1). Show top 10 rows.
import sqlite3
import pandas as pd
conn=sqlite3.connect("../data/processed/chess.db")
cursor=conn.cursor()
cursor.execute("""
SELECT 
    game_id, 
    white_id, 
    white_rating,
    RANK() OVER (
        PARTITION BY white_id 
        ORDER BY white_rating DESC
    ) AS rating_rank
FROM games
ORDER BY white_id, rating_rank
LIMIT 10;
 
    """)

cursor.execute(query)
result=cursor.fetchall()
print("11 -",result)
conn.close()

11 - [('taranga', 34), ('ssf7', 29), ('hassan1365416', 28), ('a_p_t_e_m_u_u', 25), ('1240100948', 22)]


In [ ]:
#12 LAG: show each game's white_rating and the previous game's white_rating for the same
# player. Filter to players with 5+ games.
import sqlite3
import pandas as pd
conn=sqlite3.connect("../data/processed/chess.db")
cursor=conn.cursor()
cursor.execute("""
WITH player_history AS (
    SELECT 
        white_id,
        white_rating,
        LAG(white_rating) OVER (
            PARTITION BY white_id 
            ORDER BY game_id
        ) AS prev_rating,
        COUNT(*) OVER (
            PARTITION BY white_id
        ) AS game_count
    FROM games
)
SELECT 
    white_id, 
    white_rating, 
    prev_rating
FROM player_history
WHERE game_count >= 5
ORDER BY white_id, white_rating;
 
    """)

cursor.execute(query)
result=cursor.fetchall()
print("12 -",result)
conn.close()

12 - [('taranga', 34), ('ssf7', 29), ('hassan1365416', 28), ('a_p_t_e_m_u_u', 25), ('1240100948', 22)]


#### Question Answers 
 1- [(20058, 16155)]

 2- [('Resign', 11147), ('Mate', 6325), ('Out of Time', 1680), ('Draw', 906)]

 3- [(11555, 'White', 349), (13860, 'White', 349), (16387, 'Draw', 259), (4237, 'Draw', 255), (16646, 'Draw', 226), (15479, 'Draw', 222), (16944, 'Black', 222), (6777, 'Draw', 221), (13231, 'Black', 218), (13556, 'Draw', 216)]

4- [('Black', 45.403330342008175), ('Draw', 4.736264831987237), ('White', 49.86040482600459)]

5- [('Draw', 83.78145695364239, 259), ('Out of Time', 72.74285714285715, 349), ('Mate', 65.41501976284584, 222), ('Resign', 53.91253251996053, 218)]

6- [('A00', 1007), ('C00', 844), ('D00', 739), ('B01', 716), ('C41', 691)]

7- [('St. George Defense', 1455), ("Queen's Pawn Game: Chigorin Variation", 1262), ('Borg Defense: Borg Gambit', 1229), ('Ware Opening', 1007), ("Van't Kruijs Opening", 1007)]

8 - Players never played as White: 6197

9-[('taranga', 34), ('ssf7', 29), ('hassan1365416', 28), ('a_p_t_e_m_u_u', 25), ('1240100948', 22)]

10- [('taranga', 34), ('ssf7', 29), ('hassan1365416', 28), ('a_p_t_e_m_u_u', 25), ('1240100948', 22)]

11 - [('taranga', 34), ('ssf7', 29), ('hassan1365416', 28), ('a_p_t_e_m_u_u', 25), ('1240100948', 22)]

12-12 - [('taranga', 34), ('ssf7', 29), ('hassan1365416', 28), ('a_p_t_e_m_u_u', 25), ('1240100948', 22)]




### Reset DB Pipline 


In [30]:
import sqlite3

def reset_database(db_path):
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()
    
    # 1. Turn off Foreign Key checks so we can drop tables in any order
    cursor.execute('PRAGMA foreign_keys = OFF')
    
    # 2. Drop existing tables
    # List them in order of importance
    cursor.execute('DROP TABLE IF EXISTS openings')


    conn.commit()
    conn.close()
    print("Database cleaned and schema recreated.")

# Run this at the start of your development workflow
reset_database('../data/processed/chess.db')

Database cleaned and schema recreated.


## Stage 5 — Indexes & EXPLAIN QUERY PLAN

In [7]:
import sqlite3

DB_PATH = "../data/processed/chess.db"
conn = sqlite3.connect(DB_PATH)
cursor = conn.cursor()

# Sample query that benefits from indexes on player/opening columns
sample_query = """
SELECT game_id, winner, turns
FROM games
WHERE white_id = 'taranga'
  AND opening_code = 'A00'
"""

print("=== BEFORE indexes ===")
for row in cursor.execute(f"EXPLAIN QUERY PLAN {sample_query}"):
    print(row)

# Drop indexes if re-running this cell
for idx in ("idx_games_white_id", "idx_games_black_id", "idx_games_opening_code"):
    cursor.execute(f"DROP INDEX IF EXISTS {idx}")

cursor.execute("CREATE INDEX idx_games_white_id ON games(white_id)")
cursor.execute("CREATE INDEX idx_games_black_id ON games(black_id)")
cursor.execute("CREATE INDEX idx_games_opening_code ON games(opening_code)")
conn.commit()

print("\n=== AFTER indexes ===")
for row in cursor.execute(f"EXPLAIN QUERY PLAN {sample_query}"):
    print(row)

conn.close()

=== BEFORE indexes ===
(3, 0, 61, 'SEARCH games USING INDEX idx_games_opening_code (opening_code=?)')

=== AFTER indexes ===
(3, 0, 61, 'SEARCH games USING INDEX idx_games_opening_code (opening_code=?)')


## Bonus SQL Question

In [8]:
import os
import sqlite3

import pandas as pd

DB_PATH = "../data/processed/chess.db"
conn = sqlite3.connect(DB_PATH)

# ① Which opening code has the highest Draw rate?
q1 = pd.read_sql(
    """
    SELECT opening_code,
           ROUND(100.0 * SUM(CASE WHEN winner = 'Draw' THEN 1 ELSE 0 END) / COUNT(*), 2) AS draw_rate_pct
    FROM games
    GROUP BY opening_code
    HAVING COUNT(*) >= 50
    ORDER BY draw_rate_pct DESC
    LIMIT 1
    """,
    conn,
)
print("① Highest draw-rate opening:")
print(q1)

# ② Players who won more games as Black than as White
q2 = pd.read_sql(
    """
    WITH wins AS (
        SELECT white_id AS username,
               SUM(CASE WHEN winner = 'White' THEN 1 ELSE 0 END) AS white_wins,
               0 AS black_wins
        FROM games
        GROUP BY white_id
        UNION ALL
        SELECT black_id AS username,
               0 AS white_wins,
               SUM(CASE WHEN winner = 'Black' THEN 1 ELSE 0 END) AS black_wins
        FROM games
        GROUP BY black_id
    )
    SELECT username,
           SUM(white_wins) AS white_wins,
           SUM(black_wins) AS black_wins
    FROM wins
    GROUP BY username
    HAVING SUM(black_wins) > SUM(white_wins)
    ORDER BY black_wins - white_wins DESC
    """,
    conn,
)
print("\n② Black-win > White-win players:", len(q2))
print(q2.head(10))

# ③ For each victory_status, which opening is most common?
q3 = pd.read_sql(
    """
    WITH opening_counts AS (
        SELECT victory_status,
               opening_code,
               COUNT(*) AS games_count,
               ROW_NUMBER() OVER (
                   PARTITION BY victory_status
                   ORDER BY COUNT(*) DESC
               ) AS rn
        FROM games
        GROUP BY victory_status, opening_code
    )
    SELECT victory_status, opening_code, games_count
    FROM opening_counts
    WHERE rn = 1
    ORDER BY games_count DESC
    """,
    conn,
)
print("\n③ Most common opening per victory_status:")
print(q3)

# ④ Top-3 opening families by average turns
q4 = pd.read_sql(
    """
    SELECT opening_fullname,
           ROUND(AVG(turns), 2) AS avg_turns,
           COUNT(*) AS games_count
    FROM games
    GROUP BY opening_fullname
    ORDER BY avg_turns DESC
    LIMIT 3
    """,
    conn,
)
print("\n④ Top-3 opening families by avg turns:")
print(q4)

# ⑤ Rank each player's games by turns (longest = 1) and save CSV
q5 = pd.read_sql(
    """
    WITH player_games AS (
        SELECT white_id AS player_id, game_id, turns FROM games
        UNION ALL
        SELECT black_id AS player_id, game_id, turns FROM games
    )
    SELECT player_id,
           game_id,
           turns,
           RANK() OVER (
               PARTITION BY player_id
               ORDER BY turns DESC
           ) AS turn_rank
    FROM player_games
    ORDER BY player_id, turn_rank
    """,
    conn,
)

os.makedirs("../data/processed", exist_ok=True)
q5.to_csv("../data/processed/game_ranks.csv", index=False)
print("\n⑤ Saved game ranks:", q5.shape, "-> ../data/processed/game_ranks.csv")
print(q5.head(10))

conn.close()

① Highest draw-rate opening:
  opening_code  draw_rate_pct
0          A03          12.82

② Black-win > White-win players: 3816
          username  white_wins  black_wins
0       hick4u1219           0          10
1         seciyeli           0           7
2    vikrant_dalvi           0           4
3         sjamal99           0           4
4            lcf64           0           4
5    hknight_chess           0           4
6  cmcookiemonster           0           4
7      armourbuddy           0           4
8       andreas636           0           4
9   amazingfoxtrot           0           4

③ Most common opening per victory_status:
  victory_status opening_code  games_count
0         Resign          A00          473
1           Mate          A00          416
2    Out of Time          A00           79
3           Draw          A00           39

④ Top-3 opening families by avg turns:
                                    opening_fullname  avg_turns  games_count
0        Queen's Gambit 

## Feature Table 

In [9]:
import os
import sqlite3

import pandas as pd

DB_PATH = "../data/processed/chess.db"
conn = sqlite3.connect(DB_PATH)

features_sql = """
SELECT
    game_id,
    white_rating - black_rating AS rating_diff,
    turns,
    rated,
    opening_shortname,
    COUNT(*) OVER (
        PARTITION BY white_id
        ORDER BY game_id
        ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    ) AS white_experience,
    winner
FROM games
ORDER BY game_id
"""

features_df = pd.read_sql(features_sql, conn)
os.makedirs("../data/processed", exist_ok=True)
features_df.to_csv("../data/processed/features.csv", index=False)

print("Feature table shape:", features_df.shape)
print(features_df.head(10))
print("Saved -> ../data/processed/features.csv")

conn.close()

Feature table shape: (20058, 7)
   game_id  rating_diff  turns  rated       opening_shortname  \
0        1          309     13      0            Slav Defense   
1        2           61     16      1     Nimzowitsch Defense   
2        3           -4     61      1        King's Pawn Game   
3        4          -15     61      1       Queen's Pawn Game   
4        5           54     95      1        Philidor Defense   
5        6          248      5      0        Sicilian Defense   
6        7           97     33      1  Blackmar-Diemer Gambit   
7        8         -695      9      0     Nimzowitsch Defense   
8        9           47     66      1            Italian Game   
9       10          172    119      1    Scandinavian Defense   

   white_experience winner  
0                 1  White  
1                 1  Black  
2                 1  White  
3                 1  White  
4                 1  White  
5                 1   Draw  
6                 1  White  
7                 1 